In [1]:
!pip install google-generativeai
!pip install pyautogen

  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached protobuf-5.29.6-cp310-abi3-win_amd64.whl.metadata (592 bytes)
INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
  Using cached google_api_core-2.34.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached google_api_core-2.33.0-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is looking at multiple versions of google-api-core[grpc] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of google-api-core[grpc] to determine which version is compatible with other requirements. This cou

In [2]:
!pip install autogen

In [3]:
import google.generativeai as genai

GEMINI_API_KEY = "AQ.Ab8RN6LTuy7yy"
genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel("gemini-3.5-flash")

def gemini_chat(prompt):
    response = model.generate_content(prompt)
    return response.text


ModuleNotFoundError: No module named 'google'

In [7]:
# Define Custom Gemini Agent

In [8]:
from autogen import ConversableAgent

class GeminiAgent(ConversableAgent):
    def __init__(self, name, **kwargs):
        super().__init__(name, **kwargs)

    def generate_reply(self, messages, sender, config=None):
        prompt = "\n".join([m["content"] for m in messages])
        reply = gemini_chat(prompt)
        return {"role": "assistant", "content": reply}


In [9]:
# Initialize All Agents

In [10]:
from autogen import UserProxyAgent

user_proxy = UserProxyAgent(name="User", human_input_mode="TERMINATE")

analyst = GeminiAgent(name="AnalystBot")
swot = GeminiAgent(name="SWOTBot")
strategist = GeminiAgent(name="StrategistBot")
critic = GeminiAgent(name="CriticBot")


In [11]:
# Business Description (Input)

In [12]:
business_description = """
Our company is a 2-year-old startup that provides AI-powered resume screening software for
HR teams.
We’ve gained traction in India but are struggling to expand internationally.
The product integrates with common HR tools and uses machine learning to rank candidates.
We face growing competition and challenges in compliance and data privacy in
international markets.
"""


In [13]:
# Chain Agents via Shared Context

In [14]:
# Step 1: Ask AnalystBot to analyze the business
messages = [{"role": "user", "content": f"Analyze the following business and identify key challenges and strengths:\n\n{business_description}"}]
analyst_output = analyst.generate_reply(messages, sender="User")["content"]
print("AnalystBot Output:\n", analyst_output)

# Step 2: Pass analyst_output to SWOTBot
messages = [{"role": "user", "content": f"Based on this analysis, provide a detailed SWOT analysis:\n\n{analyst_output}"}]
swot_output = swot.generate_reply(messages, sender="User")["content"]
print("\n SWOTBot Output:\n", swot_output)

# Step 3: Pass SWOT to StrategistBot
messages = [{"role": "user", "content": f"Based on this SWOT analysis, suggest 3 business growth strategies:\n\n{swot_output}"}]
strategies_output = strategist.generate_reply(messages, sender="User")["content"]
print("\n StrategistBot Output:\n", strategies_output)

# Step 4: Ask CriticBot to critique the strategies
messages = [{"role": "user", "content": f"Critique the following strategies and suggest improvements:\n\n{strategies_output}"}]
critic_output = critic.generate_reply(messages, sender="User")["content"]
print("\n CriticBot Output:\n", critic_output)


AnalystBot Output:
 Here is an analysis of your startup’s current position, broken down by strengths, key challenges, and actionable recommendations to help you bridge the gap between regional success and global scaling.

---

### **Key Strengths**

1. **Proven Market Traction in a High-Volume Environment**
   * *Why it matters:* Success in India is a massive stress-test for HR software. India’s hiring landscape features extremely high applicant volumes per job posting. Proving that your AI can process, rank, and handle this volume effectively demonstrates **core product efficacy and scalability**.

2. **Seamless Integration Capability**
   * *Why it matters:* HR teams are notoriously reluctant to abandon their primary systems (ATS/HRIS like Greenhouse, Workday, Lever, or SAP SuccessFactors). By integrating into existing workflows, you lower the friction to adopt your tool, reducing the sales cycle and technical onboarding barriers.

3. **Value Proposition Centered on Efficiency**
   *